# CyberShield AI — Exploration des données (EDA)

Notebook d'exploration du dataset d'intrusion réseau (type CICIDS2017), utilisé pour la détection d'intrusions dans le projet CyberShield AI.

**Sommaire**
1. Chargement des données brutes
2. Déséquilibre des classes
3. Valeurs manquantes / infinies (avant nettoyage)
4. Distributions des features clés (Benign vs Attaque)
5. Corrélations entre features
6. Comparaison avant/après nettoyage


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

BASE_DIR = Path.cwd().parent  # notebooks/ -> AI/
RAW_DIR = BASE_DIR / 'datasets' / 'raw'
PROCESSED_DIR = BASE_DIR / 'datasets' / 'processed'

print('Fichiers bruts trouvés :')
for f in sorted(RAW_DIR.glob('*.parquet')):
    print(' -', f.name)


## 1. Chargement des données brutes

On charge chaque fichier séparément pour garder une vue par jour/type d'attaque, avant toute fusion.


In [ ]:
raw_files = sorted(RAW_DIR.glob('*.parquet'))
raw_data = {}

for f in raw_files:
    df = pd.read_parquet(f)
    raw_data[f.stem] = df
    print(f'{f.stem:45s} : {df.shape[0]:>10,} lignes  x  {df.shape[1]} colonnes')


In [ ]:
# Aperçu du premier fichier
first_key = list(raw_data.keys())[0]
raw_data[first_key].head()


## 2. Déséquilibre des classes

Point critique pour ce projet : certaines classes d'attaque sont extrêmement rares (Heartbleed, Infiltration, Sql Injection), ce qui impacte directement la stratégie d'entraînement (poids d'échantillon, métriques à privilégier).


In [ ]:
# Trouver la colonne Label dans chaque fichier et agréger la répartition globale
label_counts = {}

for name, df in raw_data.items():
    label_col = next((c for c in df.columns if c.strip().lower() == 'label'), None)
    if label_col is None:
        continue
    counts = df[label_col].value_counts()
    for label, count in counts.items():
        label_counts[label] = label_counts.get(label, 0) + count

label_series = pd.Series(label_counts).sort_values(ascending=False)
label_series


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
label_series.sort_values().plot.barh(ax=ax, color='#4C72B0')
ax.set_xscale('log')
ax.set_xlabel('Nombre de lignes (échelle log)')
ax.set_title("Répartition des classes — échelle logarithmique\n"
             "(indispensable ici : Benign écrase tout en échelle linéaire)")
plt.tight_layout()
plt.show()

print('\nRatio classe majoritaire / classe la plus rare :',
      f"{label_series.max() / label_series.min():,.0f}x")


## 3. Valeurs manquantes / infinies (avant nettoyage)

Les datasets CICIDS2017 bruts contiennent typiquement des valeurs infinies (division par zéro dans le calcul de certains ratios, ex: `Flow Bytes/s` quand `Flow Duration = 0`). On quantifie ça avant le nettoyage pour justifier les étapes de `clean_data.py`.


In [ ]:
quality_report = []

for name, df in raw_data.items():
    numeric_df = df.select_dtypes(include=[np.number])
    n_inf = np.isinf(numeric_df.to_numpy(dtype='float64', na_value=np.nan)).sum()
    n_missing = df.isna().sum().sum()
    n_duplicates = df.duplicated().sum()
    quality_report.append({
        'fichier': name,
        'lignes': df.shape[0],
        'valeurs_infinies': n_inf,
        'valeurs_manquantes': n_missing,
        'doublons': n_duplicates,
    })

pd.DataFrame(quality_report).set_index('fichier')


## 4. Distributions des features clés (Benign vs Attaque)

On regarde ici quelques features identifiées comme importantes lors de l'entraînement du modèle (voir `explainability/`), pour visualiser intuitivement ce qui différencie le trafic malveillant du trafic normal.

⚠️ Ajustez `KEY_FEATURES` ci-dessous si vos noms de colonnes nettoyés diffèrent (espaces vs underscores selon que vous chargez du brut ou du nettoyé).


In [ ]:
# Utilise un fichier avec un bon mélange Benign/Attaque, déjà nettoyé si disponible
clean_files = sorted(PROCESSED_DIR.glob('*_clean.parquet'))
if clean_files:
    sample_df = pd.read_parquet([f for f in clean_files if 'DDoS' in f.name][0])
else:
    sample_df = raw_data[[k for k in raw_data if 'DDoS' in k][0]]

print(sample_df.shape)
sample_df.columns.tolist()[:20]


In [ ]:
KEY_FEATURES = [
    'Flow_Duration', 'Bwd_Packet_Length_Mean', 'Fwd_Packet_Length_Mean',
    'Flow_Bytes_s', 'Flow_Packets_s', 'Init_Bwd_Win_Bytes',
]
label_col = next(c for c in sample_df.columns if c.strip().lower() == 'label')
available = [f for f in KEY_FEATURES if f in sample_df.columns]
print('Features disponibles :', available)

is_attack = sample_df[label_col] != 'Benign'

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, feature in zip(axes.flat, available):
    data = sample_df[[feature]].copy()
    data['type'] = np.where(is_attack, 'Attaque', 'Benign')
    # log1p pour lisibilité, ces features ont des distributions très étalées
    for label, color in [('Benign', '#4C72B0'), ('Attaque', '#C44E52')]:
        subset = data.loc[data['type'] == label, feature]
        subset = np.log1p(subset.clip(lower=0))
        ax.hist(subset, bins=40, alpha=0.5, label=label, color=color, density=True)
    ax.set_title(feature, fontsize=10)
    ax.legend(fontsize=8)
fig.suptitle('Distribution (log1p) — Benign vs Attaque')
plt.tight_layout()
plt.show()


## 5. Corrélations entre features

Sur un échantillon (le calcul complet sur 2M+ lignes x 77 colonnes est inutilement coûteux pour une simple inspection visuelle).


In [ ]:
numeric_cols = sample_df.select_dtypes(include=[np.number]).columns
sample_for_corr = sample_df[numeric_cols].sample(n=min(20000, len(sample_df)), random_state=42)
corr = sample_for_corr.corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax, cbar_kws={'shrink': 0.6})
ax.set_title('Corrélation entre features (échantillon)')
plt.tight_layout()
plt.show()


In [ ]:
# Paires de features très corrélées (redondantes) — candidates à la réduction de dimension
corr_pairs = corr.abs().unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1.0]  # retire la diagonale
corr_pairs.drop_duplicates().head(20)


## 6. Comparaison avant / après nettoyage

Vérifie que `clean_data.py` a bien fait son travail : plus d'infinis, plus de doublons, mêmes proportions de classes.


In [ ]:
if clean_files:
    comparison = []
    for raw_name, raw_df in raw_data.items():
        matching_clean = [f for f in clean_files if raw_name in f.name]
        if not matching_clean:
            continue
        clean_df = pd.read_parquet(matching_clean[0])
        comparison.append({
            'fichier': raw_name,
            'lignes_brutes': raw_df.shape[0],
            'lignes_nettoyées': clean_df.shape[0],
            'lignes_supprimées': raw_df.shape[0] - clean_df.shape[0],
        })
    display(pd.DataFrame(comparison).set_index('fichier'))
else:
    print("Aucun fichier *_clean.parquet trouvé dans datasets/processed — "
          "lancez clean_data.py d'abord pour cette section.")


## Conclusions de l'exploration

- **Déséquilibre extrême** : ratio majoritaire/minoritaire de plusieurs dizaines de milliers pour un — confirme la nécessité de pondérer les classes à l'entraînement plutôt que de faire confiance à l'accuracy brute.
- **Valeurs infinies présentes dans les données brutes** — le nettoyage (`clean_data.py`) est indispensable avant toute utilisation.
- *(Complétez cette section avec vos propres observations une fois les graphiques exécutés.)*
